# Simple LlamaIndex PDF Q&AThis notebook uses:- **LlamaIndex**- the same PDF file: **`metagpt.pdf`**- **`gpt-5.4-nano`** for the LLM- **`text-embedding-3-small`** for embeddingsPut `metagpt.pdf` in the same folder as this notebook, then run the cells from top to bottom.

In [ ]:
!pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai pypdf python-dotenv

In [ ]:
import osfrom dotenv import load_dotenvload_dotenv()OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")if not OPENAI_API_KEY:    raise ValueError("OPENAI_API_KEY not found. Put it in your environment or .env file.")

In [ ]:
from llama_index.core import Settingsfrom llama_index.llms.openai import OpenAIfrom llama_index.embeddings.openai import OpenAIEmbeddingSettings.llm = OpenAI(model="gpt-5.4-nano", api_key=OPENAI_API_KEY)Settings.embed_model = OpenAIEmbedding(    model="text-embedding-3-small",    api_key=OPENAI_API_KEY)print("LLM and embedding model loaded.")

In [ ]:
from llama_index.core import SimpleDirectoryReaderdocuments = SimpleDirectoryReader(    input_files=["metagpt.pdf"]   # keep the same PDF name, or replace if your filename is different).load_data()print("Loaded documents:", len(documents))

In [ ]:
from llama_index.core import VectorStoreIndexindex = VectorStoreIndex.from_documents(documents)query_engine = index.as_query_engine()print("Index and query engine ready.")

In [ ]:
response = query_engine.query("What is the summary of this paper?")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))

In [ ]:
response = query_engine.query("How do agents share information with other agents?")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))

In [ ]:
response = query_engine.query("Tell me about the ablation study results.")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))

## Optional: simple router versionUse the cells below only if you want a basic summary-vs-specific-question router.

In [ ]:
from llama_index.core import SummaryIndex, VectorStoreIndexfrom llama_index.core.tools import QueryEngineToolfrom llama_index.core.query_engine.router_query_engine import RouterQueryEnginefrom llama_index.core.selectors import LLMSingleSelectorllm = OpenAI(model="gpt-5.4-nano", api_key=OPENAI_API_KEY)embed_model = OpenAIEmbedding(model="text-embedding-3-small", api_key=OPENAI_API_KEY)summary_index = SummaryIndex.from_documents(documents)vector_index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)summary_query_engine = summary_index.as_query_engine(    response_mode="tree_summarize",    llm=llm)vector_query_engine = vector_index.as_query_engine(llm=llm)summary_tool = QueryEngineTool.from_defaults(    query_engine=summary_query_engine,    description="Useful for summarization questions.")vector_tool = QueryEngineTool.from_defaults(    query_engine=vector_query_engine,    description="Useful for specific factual questions from the PDF.")router_query_engine = RouterQueryEngine(    selector=LLMSingleSelector.from_defaults(llm=llm),    query_engine_tools=[summary_tool, vector_tool],    verbose=True)print("Router query engine ready.")

In [ ]:
response = router_query_engine.query("What is the summary of this paper?")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))

In [ ]:
response = router_query_engine.query("How do agents share information with other agents?")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))

In [ ]:
response = router_query_engine.query("Tell me about the ablation study results.")print(response)print("\nSource nodes:", len(getattr(response, "source_nodes", [])))